# db_tools quick test

This notebook exercises the `db_tools` read APIs (and optional write APIs) using the single configuration file `db_tools/settings.json`.

The write section is disabled by default and will refuse to write if the configured SQLite DB is not protected (to avoid accidental in-place edits of shared DBs).

In [1]:
import json
import sys
from pathlib import Path
from urllib.parse import urlparse

sys.dont_write_bytecode = True

def _find_repo_root(start: Path) -> Path:
    p = start
    for _ in range(12):
        if (p / 'db_tools').is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError('Could not find repo root (missing db_tools/).')

REPO_ROOT = _find_repo_root(Path().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('python:', sys.executable)
print('repo_root:', REPO_ROOT)

python: /root/miniconda3/envs/kicad/bin/python
repo_root: /root/workspace/KiCAD_MCP


In [2]:
from db_tools.settings import get_settings, get_db_scheme, resolve_sqlite_path
from db_tools.workdir import is_protected_db_path

settings = get_settings(reload=True)
scheme = get_db_scheme()

print('db_url:', settings.db_url)
print('scheme:', scheme)
print('work_dir:', str(settings.work_dir))

if scheme == 'sqlite':
    src = resolve_sqlite_path(settings.db_url)
    print('configured_sqlite_path:', src)
    print('is_protected:', is_protected_db_path(src))
else:
    print('Postgres DSN is taken from db_tools/settings.json db_url.')

db_url: sqlite:////mnt/shared/catalog/catalog.sqlite3
scheme: sqlite
work_dir: /mnt/shared/db_tools/work
configured_sqlite_path: /mnt/shared/catalog/catalog.sqlite3
is_protected: True


In [3]:
import subprocess

proc = subprocess.run(
    [sys.executable, '-B', str(REPO_ROOT / 'scripts' / 'catalog' / 'smoke_check.py')],
    capture_output=True,
    text=True,
)
print(proc.stdout)
if proc.stderr.strip():
    print(proc.stderr)
print('returncode:', proc.returncode)

{
  "db": {
    "path": "/mnt/shared/catalog/catalog.sqlite3",
    "ok": true,
    "missingObjects": [],
    "sampleLcsc": -10,
    "sampleMpn": "+10V",
    "sampleFootprint": "00300210N",
    "sampleModelName": "AP9101CAK-ATTRG1"
  },
  "ok": true
}

returncode: 0


In [ ]:
from db_tools.client import DbToolsClient

client = DbToolsClient.from_settings()

search = client.search_mpn_part('SML-D13VWT86C', limit=5)
print('search_mpn_part success:', search.get('success'), 'count:', search.get('count'))
print(json.dumps(search, indent=2)[:2000])

datasheet = client.search_datasheet('SML-D13VWT86C', 'Optocoupler_LED_Digital_Tube_Photoelectric_Device')
print('search_datasheet:', json.dumps(datasheet, indent=2)[:1000])

pinout = client.get_symbol_pinout(library='power', mpn='+10V')
print('get_symbol_pinout success:', pinout.get('success'), 'pinCount:', pinout.get('pinCount'))
print(json.dumps(pinout, indent=2)[:1200])

fps = client.search_footprints('SOIC', max_results=5)
print('search_footprints:', fps.get('success'), 'matchCount:', fps.get('matchCount'))
print(json.dumps(fps, indent=2)[:1200])

model = client.search_spice_model(name='FDMC510P-MS', library='Triode_MOS_Tube_Transistor')
print('search_spice_model found:', bool(model))
if model:
    print('model name:', model.get('name'))
    print('model library:', model.get('library'))
    print('model_content_len:', len(model.get('model_content') or ''))

## Optional write tests

These write into the configured backend. For SQLite, the code refuses to write unless the configured DB path is protected (so writes go to the working copy in `work_dir`). For Postgres, only run this against a test database.

In [ ]:
RUN_WRITE_TESTS = False

if not RUN_WRITE_TESTS:
    print('Write tests disabled. Set RUN_WRITE_TESTS=True to run.')
else:
    import time
    from db_tools.settings import get_db_scheme, get_settings, get_sqlite_db_path, resolve_sqlite_path
    from db_tools.workdir import is_protected_db_path
    from db_tools.symbols import add_symbol_entry, get_symbol_pinout
    from db_tools.spice_models import save_part_model, search_spice_model
    from db_tools.component_search.search import add_searchable_part

    scheme = get_db_scheme()
    if scheme == 'sqlite':
        src = resolve_sqlite_path(get_settings().db_url)
        if not is_protected_db_path(src):
            raise RuntimeError(
                'Refusing to write: configured SQLite DB is not protected. '
                'Add its directory to protected_roots in db_tools/settings.json or point db_url at a working copy.'
            )
        write_db = get_sqlite_db_path(for_write=True)
        print('write_db_path (working copy):', write_db)
    elif scheme in {'postgres', 'postgresql'}:
        print('Writing to Postgres DSN:', get_settings().db_url)
    else:
        raise RuntimeError(f'Unsupported db_url scheme: {scheme!r}')

    suffix = str(int(time.time()))

    # 1) Write a symbol row and read its pinout back.
    sym_mpn = f'DBTOOLS_TEST_SYMBOL_{suffix}'
    sym_lib = 'DBTOOLS_TEST'
    sym_sexp = '(symbol "DBTOOLS_TEST" (pin passive (name "A") (number "1")) (pin passive (name "B") (number "2")))'
    print(add_symbol_entry(mpn=sym_mpn, library=sym_lib, sexp=sym_sexp, overwrite=True))
    print(get_symbol_pinout(library=sym_lib, mpn=sym_mpn))

    # 2) Write a SPICE model row and read it back.
    model_name = f'DBTOOLS_TEST_MODEL_{suffix}'
    model_lib = 'DBTOOLS_TEST'
    model_content = f".SUBCKT {model_name} A BR1 A B 1k.ENDS {model_name}"
    save_part_model(name=model_name, library=model_lib, model_content=model_content, vendor_provided=False)
    print(search_spice_model(name=model_name, library=model_lib))

    # 3) Add a searchable part (uses a known IC family so it matches filters).
    part_mpn = f'DBTOOLS_TEST_PART_{suffix}'
    part_library = 'Power Management'
    part_package = 'TESTPKG'
    part_datasheet = 'https://example.com/datasheet.pdf'
    print(
        add_searchable_part(
            mpn=part_mpn,
            library=part_library,
            package=part_package,
            datasheet=part_datasheet,
            attributes={'Type': 'Test Entry'},
        )
    )
    verify = client.search_mpn_part(part_mpn, limit=3)
    print('verify search:', verify.get('success'), 'count:', verify.get('count'))
    print(json.dumps(verify, indent=2)[:1500])